# Projeto 1: Reconhecimento de Objetos com YOLO

**MS960/MT862 : Tópicos em Aprendizado de Máquina (Deep Learning)**
**Prof. João Florindo : IMECC/UNICAMP : 2026/1**

**Integrantes do grupo:**
1. João (RA: ...)
2. Integrante 2 (RA: ...)
3. Integrante 3 (RA: ...)
4. Integrante 4 (RA: ...)

Este notebook implementa o algoritmo YOLOv3 em PyTorch para detecção de objetos em imagens, com as funções de Intersecção sobre União (IoU) e Supressão Não Maximal (NMS) construídas manualmente, conforme exigido pelo enunciado.

A organização do notebook segue os quatro itens pedidos:

1. Definição da arquitetura da rede (DarkNet-53 com três cabeças de detecção).
2. Carregamento, redimensionamento e exibição das imagens.
3. Funções manuais de IoU e NMS, com testes para diferentes valores de limiar.
4. Detecção propriamente dita: número de objetos, classes, confiança e bounding boxes.

## 1. Importações e configuração

Usamos PyTorch para a definição da rede e PIL/matplotlib para manipulação e visualização das imagens. As funções manuais de IoU e NMS estão isoladas no arquivo `utils.py`, separação que também é compatível com o notebook de referência distribuído pelo professor.

In [ ]:
import os
import random
import colorsys
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

# Funcoes manuais de IoU e NMS implementadas em utils.py
from utils import manual_iou, manual_nms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de processamento: {device}")
print(f"Versao do PyTorch: {torch.__version__}")

## 2. Âncoras do YOLOv3

O YOLOv3 trabalha com três escalas de saída (13×13, 26×26 e 52×52 sobre uma entrada de 416×416). A cada célula da grade são associadas três âncoras (caixas de referência) com tamanhos pré definidos por k means sobre o COCO. As âncoras maiores ficam na escala mais grosseira e capturam objetos grandes, enquanto as menores ficam na escala mais fina e capturam objetos pequenos.

In [ ]:
YOLOV3_ANCHORS = [
    [(116, 90),  (156, 198), (373, 326)],   # Escala 13x13 (objetos grandes)
    [(30, 61),   (62, 45),   (59, 119)],    # Escala 26x26 (objetos medios)
    [(10, 13),   (16, 30),   (33, 23)]      # Escala 52x52 (objetos pequenos)
]

## 3. Carregamento, redimensionamento e exibição das imagens

O YOLO espera entradas de tamanho fixo (416×416). Em vez de simplesmente redimensionar, o que distorceria a imagem e prejudicaria a localização das caixas, aplicamos uma técnica chamada *letterbox*: a imagem é reescalada preservando a razão de aspecto e o espaço restante é preenchido com cinza neutro (128, 128, 128). Isso é importante porque caixas alongadas ou achatadas trariam problemas para a regressão das coordenadas.

Também implementamos a função `reverter_escala_caixas`, que desfaz o efeito do *letterbox* sobre as coordenadas previstas, levando as caixas finais de volta ao sistema de coordenadas da imagem original.

In [ ]:
# Le o arquivo coco.names com a lista de 80 classes
def read_classes(classes_path):
    with open(classes_path) as f:
        return [c.strip() for c in f.readlines()]

# Gera uma cor distinta para cada classe usando o espaco HSV
def generate_colors(class_names):
    hsv = [(x/len(class_names), 1.0, 1.0) for x in range(len(class_names))]
    cores = [colorsys.hsv_to_rgb(*c) for c in hsv]
    cores = [(int(r*255), int(g*255), int(b*255)) for r, g, b in cores]
    random.seed(10101)
    random.shuffle(cores)
    random.seed(None)
    return cores

# Redimensiona preservando a razao de aspecto e preenchendo com cinza
def letterbox_image(image, size=(416, 416)):
    iw, ih = image.size
    w, h = size
    escala = min(w/iw, h/ih)
    nw, nh = int(iw*escala), int(ih*escala)
    image = image.resize((nw, nh), Image.BICUBIC)
    nova = Image.new('RGB', size, (128, 128, 128))
    nova.paste(image, ((w - nw)//2, (h - nh)//2))
    return nova

# Desfaz o letterbox: converte caixas normalizadas para o sistema original
def reverter_escala_caixas(boxes, img_size, original_shape):
    iw, ih = original_shape
    w, h = img_size
    escala = min(w/iw, h/ih)
    nw, nh = int(iw*escala), int(ih*escala)
    dx = (w - nw) / 2.0 / w
    dy = (h - nh) / 2.0 / h
    sw, sh = nw/w, nh/h
    boxes[:, [0, 2]] = (boxes[:, [0, 2]] - dy) / sh   # ajusta y
    boxes[:, [1, 3]] = (boxes[:, [1, 3]] - dx) / sw   # ajusta x
    boxes[:, [0, 2]] *= ih
    boxes[:, [1, 3]] *= iw
    return boxes

# Pipeline completa: abre, aplica letterbox, normaliza e converte para tensor
def preprocess_image(img_path, model_image_size=(416, 416)):
    image = Image.open(img_path).convert('RGB')
    boxed = letterbox_image(image, model_image_size)
    arr = np.array(boxed, dtype='float32') / 255.0
    arr = arr[:, :, ::-1].copy()                     # RGB para BGR
    arr = np.transpose(arr, (2, 0, 1))               # HWC para CHW
    tensor = torch.from_numpy(arr).unsqueeze(0)      # adiciona dimensao do batch
    return image, tensor

## 4. Arquitetura da rede neural

A rede do YOLOv3 é composta por dois grandes blocos.

**Backbone (DarkNet 53).** Extrai características em múltiplas resoluções. É formada por blocos convolucionais (Convolução + BatchNorm + LeakyReLU) e por blocos residuais (ResBlocks) que somam o sinal de entrada à saída de duas convoluções, aliviando o problema do desvanecimento do gradiente.

**Cabeças de detecção (Feature Pyramid).** O YOLOv3 produz três saídas em escalas distintas. A escala mais grosseira (13×13) detecta objetos grandes; ela é interpolada e concatenada com as ativações de meio nível, e o resultado entra na escala média (26×26); o mesmo procedimento gera a escala mais fina (52×52) para objetos pequenos. Esse fluxo segue a ideia de Feature Pyramid Networks.

Cada saída tem $3 \times (5 + 80) = 255$ canais, pois cada uma das três âncoras prevê quatro coordenadas, uma confiança de objeto e a probabilidade de cada uma das 80 classes.

In [ ]:
# Bloco basico: Convolucao -> BatchNorm -> LeakyReLU
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k, s, p, bn=True):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, k, s, p, bias=not bn)
        self.bn   = nn.BatchNorm2d(out_c, eps=1e-5) if bn else None
        self.act  = nn.LeakyReLU(0.1, inplace=True) if bn else None

    def forward(self, x):
        x = self.conv(x)
        if self.bn:
            x = self.act(self.bn(x))
        return x


# Bloco residual com atalho de conexao
class ResBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = ConvBlock(channels, channels//2, 1, 1, 0)
        self.conv2 = ConvBlock(channels//2, channels, 3, 1, 1)

    def forward(self, x):
        return x + self.conv2(self.conv1(x))


# YOLOv3 com backbone DarkNet 53 e tres cabecas de deteccao
class YOLOv3(nn.Module):
    def __init__(self, num_classes=80):
        super().__init__()
        self.num_classes = num_classes

        # Backbone DarkNet 53
        self.conv1  = ConvBlock(3, 32, 3, 1, 1)
        self.layer1 = self._make_layer(32, 64, 1)
        self.layer2 = self._make_layer(64, 128, 2)
        self.layer3 = self._make_layer(128, 256, 8)   # Rota 1 (alimenta escala 3)
        self.layer4 = self._make_layer(256, 512, 8)   # Rota 2 (alimenta escala 2)
        self.layer5 = self._make_layer(512, 1024, 4)  # Final do backbone

        # Cabeca 1: escala 13x13 (objetos grandes)
        self.head1_1 = self._make_c5(1024, 512)
        self.head1_2 = self._make_yolo_head(512, 1024, num_classes)

        # Cabeca 2: escala 26x26 (objetos medios)
        self.head2_1   = ConvBlock(512, 256, 1, 1, 0)
        self.upsample1 = nn.Upsample(scale_factor=2, mode='nearest')
        self.head2_2   = self._make_c5(768, 256)        # 256 + 512 (rota 2)
        self.head2_3   = self._make_yolo_head(256, 512, num_classes)

        # Cabeca 3: escala 52x52 (objetos pequenos)
        self.head3_1   = ConvBlock(256, 128, 1, 1, 0)
        self.upsample2 = nn.Upsample(scale_factor=2, mode='nearest')
        self.head3_2   = self._make_c5(384, 128)        # 128 + 256 (rota 1)
        self.head3_3   = self._make_yolo_head(128, 256, num_classes)

    def _make_layer(self, in_c, out_c, num_blocks):
        layers = [ConvBlock(in_c, out_c, 3, 2, 1)]
        for _ in range(num_blocks):
            layers.append(ResBlock(out_c))
        return nn.Sequential(*layers)

    def _make_c5(self, in_c, out_c):
        # Bloco de cinco convolucoes intercaladas (1x1 / 3x3)
        return nn.Sequential(
            ConvBlock(in_c,    out_c,    1, 1, 0),
            ConvBlock(out_c,   out_c*2,  3, 1, 1),
            ConvBlock(out_c*2, out_c,    1, 1, 0),
            ConvBlock(out_c,   out_c*2,  3, 1, 1),
            ConvBlock(out_c*2, out_c,    1, 1, 0),
        )

    def _make_yolo_head(self, in_c, out_c, num_classes):
        # Camada final com 3 * (5 + num_classes) canais
        return nn.Sequential(
            ConvBlock(in_c, out_c, 3, 1, 1),
            nn.Conv2d(out_c, 3*(5 + num_classes), 1, 1, 0, bias=True)
        )

    def forward(self, x):
        x = self.layer2(self.layer1(self.conv1(x)))
        rota1 = self.layer3(x)
        rota2 = self.layer4(rota1)
        x = self.layer5(rota2)

        # Escala 1: 13x13
        x1_5 = self.head1_1(x)
        out1 = self.head1_2(x1_5)

        # Escala 2: 26x26
        x = self.upsample1(self.head2_1(x1_5))
        x = torch.cat([x, rota2], dim=1)        # FPN
        x2_5 = self.head2_2(x)
        out2 = self.head2_3(x2_5)

        # Escala 3: 52x52
        x = self.upsample2(self.head3_1(x2_5))
        x = torch.cat([x, rota1], dim=1)        # FPN
        x3_5 = self.head3_2(x)
        out3 = self.head3_3(x3_5)

        return out1, out2, out3

## 5. Carregamento dos pesos pré treinados

O arquivo `yolov3.weights` foi gerado pelo framework original em C (Darknet) e armazena os pesos em ordem específica: para cada bloco convolucional com BatchNorm são lidos primeiro os parâmetros do BN (bias, peso, média, variância), depois os pesos da convolução. Para as três camadas finais (sem BN) lemos primeiro o bias e depois os pesos. A função abaixo percorre os módulos do modelo nessa mesma ordem e copia os valores para os tensores correspondentes.

In [ ]:
# Le o arquivo .weights binario do Darknet e injeta nos parametros do modelo
def carregar_pesos_yolov3(caminho_weights, modelo):
    print(f"Lendo pesos de: {caminho_weights}")
    with open(caminho_weights, "rb") as f:
        _ = np.fromfile(f, dtype=np.int32, count=5)            # cabecalho de 5 inteiros
        weights = np.fromfile(f, dtype=np.float32)

    # Coleta na ordem correta os modulos com peso
    modulos = []
    for m in modelo.modules():
        if isinstance(m, ConvBlock) or (isinstance(m, nn.Conv2d) and m.bias is not None):
            modulos.append(m)

    ptr = 0
    for modulo in modulos:
        if isinstance(modulo, ConvBlock):
            conv, bn = modulo.conv, modulo.bn
            n = bn.bias.numel()
            bn.bias.data.copy_(torch.from_numpy(weights[ptr:ptr+n]).view_as(bn.bias));               ptr += n
            bn.weight.data.copy_(torch.from_numpy(weights[ptr:ptr+n]).view_as(bn.weight));           ptr += n
            bn.running_mean.data.copy_(torch.from_numpy(weights[ptr:ptr+n]).view_as(bn.running_mean)); ptr += n
            bn_var = torch.from_numpy(weights[ptr:ptr+n]).view_as(bn.running_var)
            bn.running_var.data.copy_(torch.clamp(bn_var, min=1e-5));                                ptr += n
            nw = conv.weight.numel()
            conv.weight.data.copy_(torch.from_numpy(weights[ptr:ptr+nw]).view_as(conv.weight));      ptr += nw
        elif isinstance(modulo, nn.Conv2d):
            nb = modulo.bias.numel()
            modulo.bias.data.copy_(torch.from_numpy(weights[ptr:ptr+nb]).view_as(modulo.bias));      ptr += nb
            nw = modulo.weight.numel()
            modulo.weight.data.copy_(torch.from_numpy(weights[ptr:ptr+nw]).view_as(modulo.weight));  ptr += nw

    print(f"Carregamento concluido. Total de parametros lidos: {ptr:,}")
    return modelo

## 6. Decodificação das saídas (YOLO Head)

A saída bruta da rede tem dimensão $(B, 3 \cdot (5 + 80), H, W)$ em cada escala. Cada célula prevê três caixas, e cada caixa tem:

* duas coordenadas $(t_x, t_y)$ do centro, transformadas por sigmóide para ficar no intervalo $[0, 1]$ dentro da célula;
* duas coordenadas $(t_w, t_h)$ que entram em uma exponencial multiplicada pelo tamanho da âncora;
* uma confiança de "tem objeto aqui";
* 80 probabilidades de classe (cada uma em sigmóide, pois o YOLOv3 trata as classes de forma independente, permitindo classificação multilabel).

A pontuação final de cada par caixa/classe é `confianca * prob_classe`.

In [ ]:
# Converte as saidas brutas da rede em (caixas, scores) na escala normalizada
def decode_yolo(feats, anchors, num_classes, img_size=416):
    B, C, H, W = feats.shape
    n_anchors = len(anchors)
    feats = feats.view(B, n_anchors, 5+num_classes, H, W).permute(0, 1, 3, 4, 2).contiguous()

    grid_y, grid_x = torch.meshgrid(torch.arange(H), torch.arange(W), indexing='ij')
    grid = torch.stack((grid_x, grid_y), dim=-1).float().to(feats.device).view(1, 1, H, W, 2)

    # Centro da caixa
    box_xy = torch.sigmoid(feats[..., :2])
    box_xy = (box_xy + grid) / torch.tensor([W, H], dtype=torch.float32, device=feats.device)

    # Largura e altura
    anchors_t = torch.tensor(anchors, dtype=torch.float32, device=feats.device).view(1, n_anchors, 1, 1, 2)
    box_wh = torch.exp(torch.clamp(feats[..., 2:4], max=15.0)) * anchors_t
    box_wh = box_wh / img_size

    # Confianca e classes
    box_confidence  = torch.sigmoid(feats[..., 4:5])
    box_class_probs = torch.sigmoid(feats[..., 5:])

    # Conversao para o formato [y1, x1, y2, x2]
    box_mins  = box_xy - (box_wh/2.)
    box_maxes = box_xy + (box_wh/2.)
    boxes = torch.cat([box_mins[..., 1:2], box_mins[..., 0:1],
                       box_maxes[..., 1:2], box_maxes[..., 0:1]], dim=-1)
    scores = box_confidence * box_class_probs
    return boxes.view(-1, 4), scores.view(-1, num_classes)

## 7. IoU e NMS manuais

As implementações estão em `utils.py`. Recapitulando a ideia:

**Intersecção sobre União (IoU).** Dadas duas caixas, a IoU é a razão entre a área de interseção e a área de união. Vale 1 quando as caixas coincidem e 0 quando são disjuntas. É a métrica que mede quão parecidas são duas caixas em termos de localização.

**Supressão Não Maximal (NMS).** Como a rede produz centenas de candidatas (cada célula propõe três caixas, em três escalas), muitas delas se referem ao mesmo objeto. O NMS remove redundâncias: ordena as caixas por confiança, mantém a de maior pontuação e descarta todas as outras que tenham IoU acima de um limiar com ela. Importante: como o professor reforçou na errata, o NMS deve ser executado independentemente por classe, pois faz sentido manter duas caixas sobrepostas se forem classes diferentes (por exemplo, uma pessoa em cima de uma moto).

A célula abaixo apenas mostra o código completo das funções importadas para conferência.

In [ ]:
# Codigo das funcoes manuais (definidas em utils.py)
import inspect
print(inspect.getsource(manual_iou))
print(inspect.getsource(manual_nms))

## 8. Pipeline de detecção e desenho

Reunimos todas as etapas em duas funções: `detectar` faz inferência, filtra e aplica o NMS manual; `desenhar_deteccoes` produz a imagem com os bounding boxes.

In [ ]:
# Executa toda a cadeia de deteccao para uma imagem
def detectar(image_file, model, class_names, device,
             score_threshold=0.5, iou_threshold=0.45, max_boxes=20):
    image, image_data = preprocess_image(image_file, (416, 416))
    image_data = image_data.to(device)

    model.eval()
    with torch.no_grad():
        out1, out2, out3 = model(image_data)
        b1, s1 = decode_yolo(out1, YOLOV3_ANCHORS[0], len(class_names), 416)
        b2, s2 = decode_yolo(out2, YOLOV3_ANCHORS[1], len(class_names), 416)
        b3, s3 = decode_yolo(out3, YOLOV3_ANCHORS[2], len(class_names), 416)

        all_boxes  = torch.cat([b1, b2, b3], dim=0)
        all_scores = torch.cat([s1, s2, s3], dim=0)

        # Remove eventuais NaN/Inf decorrentes de exp em valores grandes
        valid = torch.isfinite(all_boxes).all(dim=-1) & torch.isfinite(all_scores).all(dim=-1)
        all_boxes  = all_boxes[valid]
        all_scores = all_scores[valid]

        # Filtragem por confianca
        box_class_scores, box_classes = torch.max(all_scores, dim=-1)
        mask = box_class_scores >= score_threshold
        boxes   = all_boxes[mask]
        scores  = box_class_scores[mask]
        classes = box_classes[mask]

        if boxes.size(0) == 0:
            return image, torch.empty(0, 4), torch.empty(0), torch.empty(0, dtype=torch.long)

        # Volta ao sistema de coordenadas da imagem original
        boxes = reverter_escala_caixas(boxes, (416, 416), image.size)

        # NMS manual com tratamento por classe (errata aplicada)
        keep = manual_nms(boxes, scores, classes, iou_threshold)
        keep = keep[:max_boxes]

        boxes, scores, classes = boxes[keep], scores[keep], classes[keep]

    return image, boxes, scores, classes


# Desenha caixas e rotulos sobre uma copia da imagem original
def desenhar_deteccoes(image, boxes, scores, classes, class_names, colors=None):
    image = image.copy()
    if colors is None:
        colors = generate_colors(class_names)
    font = ImageFont.load_default()
    espessura = max(2, (image.size[0] + image.size[1]) // 300)

    for i, c in reversed(list(enumerate(classes.cpu().numpy()))):
        nome  = class_names[int(c)]
        box   = boxes[i].cpu().numpy()
        score = scores[i].cpu().item()

        rotulo = f'{nome} {score:.2f}'
        draw = ImageDraw.Draw(image)
        bbox = draw.textbbox((0, 0), rotulo, font=font)
        rotulo_size = (bbox[2]-bbox[0], bbox[3]-bbox[1])

        top, left, bottom, right = box
        top    = max(0, int(np.floor(top + 0.5)))
        left   = max(0, int(np.floor(left + 0.5)))
        bottom = min(image.size[1], int(np.floor(bottom + 0.5)))
        right  = min(image.size[0], int(np.floor(right + 0.5)))
        if left >= right or top >= bottom:
            continue

        text_origin = np.array([left, top - rotulo_size[1]]) if top - rotulo_size[1] >= 0 else np.array([left, top + 1])

        for j in range(espessura):
            if left+j >= right-j or top+j >= bottom-j:
                break
            draw.rectangle([left+j, top+j, right-j, bottom-j], outline=colors[int(c)])

        draw.rectangle([tuple(text_origin), tuple(text_origin + rotulo_size)], fill=colors[int(c)])
        draw.text(tuple(text_origin), rotulo, fill=(0, 0, 0), font=font)
        del draw
    return image


# Imprime no terminal o numero, a classe e a confianca de cada deteccao
def imprimir_deteccoes(boxes, scores, classes, class_names):
    n = len(boxes)
    print(f"Total de objetos detectados: {n}")
    for i in range(n):
        nome   = class_names[int(classes[i])]
        conf   = float(scores[i])
        y1, x1, y2, x2 = boxes[i].tolist()
        print(f"  {i+1:>2}. {nome:<14}  confianca = {conf:.3f}   "
              f"bbox = (x1={x1:.0f}, y1={y1:.0f}, x2={x2:.0f}, y2={y2:.0f})")

## 9. Instanciação do modelo e carga de pesos

A primeira vez que rodamos, lemos o arquivo binário `yolov3.weights` e salvamos um `.pth` para acelerar execuções futuras. Caso o arquivo `.pth` já exista, ele é simplesmente recarregado.

In [ ]:
class_names = read_classes("data/coco.names")
print(f"Numero de classes COCO: {len(class_names)}")

modelo = YOLOv3(num_classes=len(class_names)).to(device)

caminho_pth = "yolov3_convertido.pth"
if os.path.exists(caminho_pth):
    modelo.load_state_dict(torch.load(caminho_pth, map_location=device, weights_only=True))
    print("Pesos carregados do .pth.")
else:
    modelo = carregar_pesos_yolov3("weights/yolov3.weights", modelo)
    torch.save(modelo.state_dict(), caminho_pth)
    print("Pesos salvos em yolov3_convertido.pth para uso futuro.")

modelo.eval()
total_params = sum(p.numel() for p in modelo.parameters())
print(f"Parametros totais do modelo: {total_params:,}")

## 10. Exibição da arquitetura da rede (Item 1 do enunciado)

A representação completa do `nn.Module` é grande, então mostramos um resumo agregado e em seguida a estrutura por blocos. O modelo segue exatamente o esquema 53 + 53 do DarkNet 53 com três cabeças piramidais.

In [ ]:
# Resumo agregado
n_conv = sum(1 for m in modelo.modules() if isinstance(m, nn.Conv2d))
n_bn   = sum(1 for m in modelo.modules() if isinstance(m, nn.BatchNorm2d))
n_res  = sum(1 for m in modelo.modules() if isinstance(m, ResBlock))

print("=" * 60)
print("RESUMO DA ARQUITETURA DO YOLOv3")
print("=" * 60)
print(f" Convolucoes 2D ............ {n_conv}")
print(f" BatchNorm2d ............... {n_bn}")
print(f" Blocos residuais .......... {n_res}")
print(f" Parametros totais ......... {total_params:,}")
print(f" Tamanho da entrada ........ 3 x 416 x 416")
print(f" Saidas (3 escalas) ........ 13x13, 26x26, 52x52")
print(f" Canais por saida .......... 3 * (5 + {len(class_names)}) = {3*(5+len(class_names))}")
print("=" * 60)

# Estrutura por blocos
print("\nESTRUTURA POR BLOCOS\n")
print(modelo)

## 11. Carregamento, exibição e detecção em uma imagem (Itens 2 e 4)

Aplicamos o pipeline completo na imagem `dog.jpg`, que aparece no enunciado como referência. A célula imprime as detecções (classe, confiança e bounding box) e em seguida exibe a imagem anotada.

In [ ]:
img_teste = "images/dog.jpg"

image, boxes, scores, classes = detectar(
    img_teste, modelo, class_names, device,
    score_threshold=0.5, iou_threshold=0.45)

imprimir_deteccoes(boxes, scores, classes, class_names)

saida = desenhar_deteccoes(image, boxes, scores, classes, class_names)
plt.figure(figsize=(10, 10))
plt.imshow(saida)
plt.axis('off')
plt.title("Detecção em dog.jpg (score=0.5, IoU=0.45)")
plt.show()

## 12. Experimentos com diferentes limiares (Item 3)

Variamos a IoU usada no NMS sobre a imagem `horses.jpg`, que tem quatro cavalos parcialmente sobrepostos. Esperamos que valores muito baixos de IoU sejam agressivos demais e suprimam cavalos legítimos, enquanto valores muito altos deixem várias caixas redundantes sobre o mesmo cavalo.

In [ ]:
img_alvo = "images/horses.jpg"
limiares_iou = [0.10, 0.30, 0.45, 0.60, 0.90]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, iou_t in enumerate(limiares_iou):
    image, boxes, scores, classes = detectar(
        img_alvo, modelo, class_names, device,
        score_threshold=0.5, iou_threshold=iou_t, max_boxes=50)
    saida = desenhar_deteccoes(image, boxes, scores, classes, class_names)
    axes[i].imshow(saida)
    axes[i].axis('off')
    axes[i].set_title(f"IoU = {iou_t}  ({len(boxes)} caixas)")

axes[5].axis('off')
plt.suptitle("Efeito do limiar de NMS em horses.jpg (score=0.5)", fontsize=14)
plt.tight_layout()
plt.show()

Agora variamos o limiar de score (confiança mínima) mantendo a IoU fixa em 0.45. Como esperamos pouco efeito quando todas as detecções são confiantes (caso dos cavalos), repetimos o teste em `dog2.jpg`, que contém detecções de várias confianças.

In [ ]:
img_alvo2 = "images/dog2.jpg"
limiares_score = [0.20, 0.40, 0.60, 0.80]

fig, axes = plt.subplots(1, 4, figsize=(20, 6))
for i, s_t in enumerate(limiares_score):
    image, boxes, scores, classes = detectar(
        img_alvo2, modelo, class_names, device,
        score_threshold=s_t, iou_threshold=0.45, max_boxes=50)
    saida = desenhar_deteccoes(image, boxes, scores, classes, class_names)
    axes[i].imshow(saida)
    axes[i].axis('off')
    axes[i].set_title(f"score = {s_t}  ({len(boxes)} caixas)")

plt.suptitle("Efeito do limiar de confianca em dog2.jpg (IoU=0.45)", fontsize=14)
plt.tight_layout()
plt.show()

## 13. Detecção em todas as imagens da base

Por fim, aplicamos o detector com configuração padrão (`score=0.5`, `IoU=0.45`) em todas as imagens fornecidas. Os resultados quantitativos discutidos no relatório foram coletados desta seção.

In [ ]:
imagens = sorted([f for f in os.listdir("images") if f.lower().endswith((".jpg", ".jpeg", ".png"))])
print(f"Total de imagens na base: {len(imagens)}\n")

resumo = {}
for nome in imagens:
    caminho = os.path.join("images", nome)
    image, boxes, scores, classes = detectar(
        caminho, modelo, class_names, device,
        score_threshold=0.5, iou_threshold=0.45)
    classes_lista = [class_names[int(c)] for c in classes]
    resumo[nome] = {"n": len(boxes), "classes": classes_lista}
    print(f"{nome:<18} -> {len(boxes)} obj : {classes_lista}")

# Visualizacao em grade
n = len(imagens)
cols = 3
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
axes = axes.flatten()
for ax, nome in zip(axes, imagens):
    caminho = os.path.join("images", nome)
    image, boxes, scores, classes = detectar(
        caminho, modelo, class_names, device,
        score_threshold=0.5, iou_threshold=0.45)
    saida = desenhar_deteccoes(image, boxes, scores, classes, class_names)
    ax.imshow(saida)
    ax.axis('off')
    ax.set_title(f"{nome} ({len(boxes)} obj.)")
for ax in axes[len(imagens):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Encerramento

Todas as etapas pedidas no enunciado foram cobertas:

* item 1: arquitetura completa exibida na seção 10;
* item 2: pipeline de carregamento, redimensionamento via *letterbox* e exibição implementada nas seções 3 e 11;
* item 3: experimentos com diferentes limiares de NMS e de score na seção 12;
* item 4: detecção com classes, confianças e bounding boxes nas seções 11 e 13.

A discussão crítica dos resultados, das hipóteses sobre os erros e das melhorias possíveis está no relatório que acompanha este notebook.